# Biomolecular simulation with GROMACS

In this tutorial, we will go over the very basics of standard biomolecular simulation. For that, we will use [GROMACS](https://www.gromacs.org/), one of the most widely used MD engines when it comes to biomolecules. GROMACS is a highly optimised package, with literal decades of development and the first choice for thousands of researches around the world. This tutorial will NOT be an exhaustive look into everything one can do with GROMACS, since the ecosystem is huge and can take years to master completely. Instead, we will familiarise ourselves with its basic command line interface and, hopefully, in the end we will be able to comfortably set up simple systems and run unrestrained MD simulations.

## Part 1 - Preparing a protein simulation

In the first part of the tutorial we will learn how to prepare a small protein-in-water system and get it ready for a production simulation. We will start from a PDB file containing only the protein and end up with some GROMACS binaries ready to run.

### 1.1 - Familiarising ourselves with the system

The very first thing we should really do is visualize the system that we want to simulate. 

The following display is interactive, play with it for a while. What kind of molecule are we dealing with? What can you tell me about it? Why does it "move" if it is only one PDB file?

In [1]:
import MDAnalysis as mda
import nglview as nv

u_2RVD = mda.Universe("structures/2RVD.pdb")
nv.show_mdanalysis(u_2RVD)

/home/xnadre/.local/share/mamba/envs/gromacs-tutorial/lib/python3.12/site-packages/nglview/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


NGLWidget(max_frame=19)

It is probably a good idea we inspect the [PDB file](structures/2RVD.pdb). Open it in another tab. You will see ato some point these two lines:

```text
KEYWDS    BETA-HAIRPIN, MINI-PROTEIN, CHIGNOLIN, DE NOVO PROTEIN                
EXPDTA    SOLUTION NMR
```

Here we actually have some answers to the previous questions! We are dealing with a *de novo* designed protein, a beta-hairpin, studied through solution NMR. A bit further down we will read:

```text
REMARK 210 BEST REPRESENTATIVE CONFORMER IN THIS ENSEMBLE : 1
```

So this is a conformational ensemble! That explains why the molecule "moves" in the visualiser, every frame corresponds to one conformation. We are also told that the most representative conformer is the first one. Probably, then, it is a good idea if we save it.

In [2]:
u_2RVD.trajectory[0] # Conformers are stored as if they were a trajectory
u_2RVD.atoms.write("exercise-1/2RVD_conf1.pdb")

u_conf1 = mda.Universe("exercise-1/2RVD_conf1.pdb")
nv.show_mdanalysis(u_conf1)


/home/xnadre/.local/share/mamba/envs/gromacs-tutorial/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:885: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn(
/home/xnadre/.local/share/mamba/envs/gromacs-tutorial/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(
/home/xnadre/.local/share/mamba/envs/gromacs-tutorial/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:479: UserWarning: 1 A^3 CRYST1 record, this is usually a placeholder. Unit cell dimensions will be set to None.
  warnings.warn(


NGLWidget()

Now we can see that there is no play button on the visualizer, and have just one structure. This will be our starting point and we are now ready to really start thinking about setting up the simulation.

### 1.2 - Preparation of the structure

Let's think now about where does this protein live. Is it in vaccuum? Does it occupy any space at all? Surely, it must, right? Luckily PDB files provide us with that kind of information. If we keep reading the PDB file we will bump into these lines:

```text
REMARK 215 THE COORDINATES IN THIS ENTRY WERE GENERATED FROM SOLUTION           
REMARK 215 NMR DATA.  PROTEIN DATA BANK CONVENTIONS REQUIRE THAT                
REMARK 215 CRYST1 AND SCALE RECORDS BE INCLUDED, BUT THE VALUES ON              
REMARK 215 THESE RECORDS ARE MEANINGLESS.  
```

What this means, really, is that there is really no unit-cell information for this system, this is, for all effects the protein is sitting in vacuum in an infinitely large box. That is not desirable for a condensed matter system. We would like our protein to be contained in a finite volume and be solvated by something.

Let's start with putting our protein in a finite box. This will be the first GROMACS command we will run. Open a terminal, and type the following

```bash
gmx editconf -f exercise-1/2RVD_conf1.pdb -o exercise-1/2RVD_box.pdb -box 5 5 5
```

`editconf` means edit conformation, and is a command that allows simple manipulations of structure files, like translations, rotations and, in this case, adding a simulation box. We use `-f` to indicate the input file, `-o` for an output file and `-box` to give the size of the box in nanometres. By default, the box is an ortorrhombic box, so the angles between the box vectors will be 90 degrees. `editconf` can take many more options, you can inspect all you can do with it if you run `gmx editconf -h`; and this is true for all available GROMACS commands.

We can now visualise the result of the previous operation.

In [3]:
u_box = mda.Universe("exercise-1/2RVD_box.pdb")

view = nv.show_mdanalysis(u_box)
view.add_cartoon(selection="protein")
view.add_ball_and_stick(selection="protein")
view.add_unitcell()
view.center()
view

NGLWidget()

We now see a box around our protein! Now GROMACS will understand that the protein must live at all times in that volume and does not have (in principle) an infinite space to go around. We will see in a bit why I said *in principle*. 

However, our protein still looks quite lonely in the middle of that box. In nature, proteins usually do not appear in vacuum, unless something goes really, *really* wrong. We must therefore fix this and **solvate** the protein. The solvent of choice will be, of course, water; since that is the solvent of life. We can use GROMACS for that, in the same terminal you opened before, run:

```bash
gmx solvate -cp exercise-1/2RVD_box.pdb -o exercise-1/2RVD_solv.pdb
```

In this case `solvate` is pretty much self-explanatory. `-cp` stands for Conformation of the Protein and is used to pass the conformation of the thing to be solvated. Despite its name, you do NOT need to pass necessarily a protein, it can really be anything. Again, with `-o` we decide the output file.

We can visualise the result of this:

In [4]:
u_solv = mda.Universe("exercise-1/2RVD_solv.pdb")

view = nv.show_mdanalysis(u_solv)
view.clear_representations()
view.add_cartoon(selection="protein")
view.add_ball_and_stick(selection="resname SOL")
view.add_unitcell()
view

/home/xnadre/.local/share/mamba/envs/gromacs-tutorial/lib/python3.12/site-packages/MDAnalysis/topology/PDBParser.py:372: UserWarning: Unknown element  found for some atoms. These have been given an empty element record. If needed they can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn(wmsg)


NGLWidget()

Now, that looks much better! We have quite a few waters accompanying our protein now. However, you may notice something that could seem strange at first: why are some water molecules **outside** the simulation box?

The answer to that question is that they actually aren't! In condensed-matter simulation we will work, more often than not, under Periodic Boundary Conditions (PBC). Think about pac-man, he comes out one side of the map and immediately appears again through the wall in the opposite side. This is exactly what is happening here, the water molecules that appear outside the box are actually inside, just on the opposite side. This is, partly, why I mentioned before that the atoms have *in principle* a finite volume to move. The **are** constrained within a simulation box, but that doesn't mean they will bounce off its limits, they can actually diffuse freely as much as they want. Moreover, the simulation boxed do not necessarily stay the same, but can (and usually should!) change as the simulation progresses. Again, more on this later.

With this, we have now a much more decent system and we can move to the next part of the tutorial.

### 1.3 - Parameter assignement

So far, we have just sort of manually placed molecules and atoms in a simulation box, but we have not run any simulation yet; no energies, no forces, no dynamics, no movement whatsoever. We must raise the question, however, of how will we inform our simulation engine about how to actually simulate our system. First and foremost, thus, we must decide our force field. 

*Force field* is just a fancy term to call the set equations that will be used to describe our system. Think, for example, how can we describe a bond between two particles, maybe with a harmonic potential? What about electrostatic interactions, surely we would use the Coulomb potential? And dispersion-repulsion forces, maybe the Lennard-Jones or something else? All those choices consitute the force field. Force fields come in several families which depend on the specific equations used to describe the system, their spatial resolution, etc. 

For this case, we will stick to a classical, atomistic force field; we will use a set of equations (this is, a functional form) that assigns one interacting particle per atom in the system, thus atomistic; and using physical laws completely contained within the classical formulation, with no quantum mechanics whatsoever. The advantage of these types of force fields is that they are relatively cheap to compute, at the loss of some accuracy in the treatment of the interactions. Despite this, they are widely used in biomolecular simulation.

We can use GROMACS to easily assign force field parameters to a structure file. In your terminal, execute the following:

```bash
gmx pdb2gmx -f exercise-1/2RVD_solv.pdb -o exercise-1/2RVD_amber.pdb -p exercise-1/2RVD_topol.top -i exercise-1/2RVD_posre.itp -ignh
```

In this command `pdb2gmx` takes us from a PDB file to GROMACS-usable inputs. It will create almost all that is necessary for us to run our subsequent simulations. With `-p` we will select the file to which GROMACS will write the *topology* of the system, basically a file that contains all the necessary definitions for the engine to understand *what* we are simulating and *how* we are doing it. With `-i` we indicate to which file to write some extra includes that the topology may need. `-ignh` stands for Ignore Hydrogens, this will basically automatically assign Hydrogen connectivity to heavy atoms based on geometry, and rename them; useful because sometimes the nomenclature of hydrogens in PDB files can be non-compliant with GROMACS standards.

You will be prompted first to select which force field do you want to use. GROMACS can automatically assign parameters using several force fields. For our case, we will use AMBER99SB-ILDN, so select `8` and press enter. Immediately after this, we will be asked about what water model to use. TIP3P will suffice for us, so select `1` and press enter again. 

Let us now pay attention to some extra bits of information that GROMACS gives us when we run this. GROMACS commands tend to be quite verbose and provide lots of info, it is recommended to read them carefully the very first times we are running them to understand what we are doing. It will mention how it is opening force field file definitions, analyse the PDB file, split residues, assign parameters for bonds, angles, torsions, etc. Nearly at the end we will encounter: 

```text
Total mass in system 73356.314 a.m.u.

Total charge in system -2.000 e
```

What do you think of these numbers? Do they seem reasonable to you? Should we do something about this? We will come back to this in a bit.

For now, let's open and read the generated topology file, [`exercise-1/2RVD_topol.top`](exercise-1/2RVD_topol.top). The file starts with several comments on how it was generated (note how comments are indicated with `;` characters). It then proceeds to `#include` the master force field file, where the main definitions are located, think of, for example, what are the van der Waals parameters for each atom type. It then proceeds to define several sections, like `[ atoms ]` or `[ bonds ]` or `[ angles ]`. GROMACS is, unfortunately, a bit cryptic with these definitions and can be difficult to understand for someone who just got started. The only thing you need to know now is that, in general, they define for which atoms (by their index) a given interaction happens, which functional form is used to describe such interaction (usually under the colums `funct`) and some extra data. For a more detailed explanation you can consult the [GROMACS documentation](https://manual.gromacs.org/documentation/current/reference-manual/topologies/topology-file-formats.html).

### 1.4 - Energy minimisation

